In [39]:
import requests
from bs4 import BeautifulSoup
import nltk
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import ssl
import pandas as pd
import os
from sklearn.model_selection import train_test_split

In [2]:
nltk.download("stopwords")
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Jon\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Jon\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Jon\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
icd_9_codes_wiki = requests.get("https://en.wikipedia.org/wiki/List_of_ICD-9_codes_E_and_V_codes:_external_causes_of_injury_and_supplemental_classification")
soup = BeautifulSoup(icd_9_codes_wiki.text, "html.parser")



In [4]:
tds_with_nowrap = soup.find_all('td')

code_mapping = dict()
category_descs_concat = dict()

for td in tds_with_nowrap:
    span = td.find('span', class_='nowrap')
    if span:
        try:
            split = span.a['title'].split(":")
            title = split[1][1:]
            codes = split[0]
            codes = codes.split(" ")[4].split("–")
            for i in range(int(codes[0]), int(codes[1]) + 1):
                code_mapping[i] = title
            category_descs_concat[title] = ""
        except:
            pass

external_str = "external causes of injury"
supp_str = "supplementary"
category_descs_concat[external_str] = ""
category_descs_concat[supp_str] = ""

In [5]:
with open('data/CMS32_DESC_LONG_DX.txt', 'r', encoding='latin1') as file:
    lines = [line.strip() for line in file]

In [6]:
for line in lines:
    split = line.split(" ")
    code = split[0][:3]
    desc = " ".join(split[2:])
    if code[0] == "E":
        category = external_str
    elif code[0] == "V":
        category = supp_str
    else:
        category = code_mapping[int(code)]
    category_descs_concat[category] += desc + " "

In [7]:
stop_words = set(stopwords.words('english'))

In [8]:
final_output = dict()
num_most_common = 20


for category in category_descs_concat:
    concat = category_descs_concat[category]
    tokenized = words = word_tokenize(concat.lower())
    tokenized = [word for word in tokenized if word not in stop_words and word not in string.punctuation]
    word_freq = Counter(tokenized)
    most_common = word_freq.most_common(num_most_common)
    most_common_lst = [word for word, _ in most_common]
    most_common_str = ' '.join(most_common_lst)
    final_output[category] = most_common_str

In [9]:
df = pd.DataFrame.from_dict(final_output, orient="index")

if not os.path.exists("data"):
    os.mkdir("data")
df.to_csv("data/mined_descriptions.csv")

In [10]:
df

,0
infectious and parasitic diseases,found unspecified bacilli bacteriological exam...
neoplasms,neoplasm malignant lymph nodes unspecified sit...
"endocrine, nutritional and metabolic diseases, and immunity disorders",type unspecified uncontrolled disorders defici...
diseases of the blood and blood-forming organs,unspecified anemia disease deficiency specifie...
mental disorders,disorder unspecified type episode remission de...
diseases of the nervous system and sense organs,unspecified eye without vision migraine disord...
diseases of the circulatory system,unspecified disease heart infarction chronic a...
diseases of the respiratory system,due unspecified pneumonia acute chronic respir...
diseases of the digestive system,obstruction unspecified without mention hemorr...
diseases of the genitourinary system,unspecified specified lesion chronic kidney fe...


In [12]:
icd = pd.read_csv('data/DIAGNOSES_ICD.csv')
notes = pd.read_csv('data/NOTEEVENTS.csv')

C:\Users\Jon\AppData\Local\Temp\ipykernel_21112\1014374558.py:2: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  notes = pd.read_csv('data/NOTEEVENTS.csv')


In [34]:
merged = pd.merge(notes, icd, on=["SUBJECT_ID", "HADM_ID"])
merged = merged.dropna(subset=['TEXT', 'ICD9_CODE'])

In [35]:
category_to_number_mapping = {}
count = 0
for key in final_output:
    category_to_number_mapping[key] = count
    count += 1
def map_labels(icd9_code):
    code = icd9_code[:3]
    if code[0] == "E":
        category = external_str
    elif code[0] == "V":
        category = supp_str
    else:
        category = code_mapping[int(code)]
    return category_to_number_mapping[category]

In [36]:
merged['label'] = merged['ICD9_CODE'].apply(map_labels)

In [48]:
data = merged[['TEXT', 'label']]
data = data.drop_duplicates(subset=['TEXT', 'label'])
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
train_data, test_data = train_test_split(data, test_size=0.5, random_state=42)

In [41]:
with open('data/train.txt', 'w', encoding='utf-8') as f_text, \
     open('data/train_labels.txt', 'w', encoding='utf-8') as f_label:
    for text, label in zip(train_data['TEXT'], train_data['label']):
        f_text.write(text.strip().replace('\n', ' ') + '\n')
        f_label.write(str(label) + '\n')

In [42]:
with open('data/test.txt', 'w', encoding='utf-8') as f_text, \
     open('data/test_labels.txt', 'w', encoding='utf-8') as f_label:
    for text, label in zip(test_data['TEXT'], test_data['label']):
        f_text.write(text.strip().replace('\n', ' ') + '\n')
        f_label.write(str(label) + '\n')

In [49]:
len(merged)

25735947

In [44]:
len(icd)

651047

In [46]:
len(notes)

2083180

In [55]:
len(set(data['TEXT']))

1801852

In [56]:
1801852 / len(notes)

0.8649526205128697